# DATA 266 HW1 Neural Networks

In [ ]:
import os
import random
import numpy as np
import torch
import tensorflow as tf

SID4 = 9486
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
tf.random.set_seed(SEED)

print(f"SID4  = {SID4}")
print(f"SEED  = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}")
print(f"CLS_A = {CLS_A}")
print(f"CLS_B = {CLS_B}")


In [ ]:
BASELINE_CONFIG = {
    "hidden_layers": [64, 32],
    "learning_rate": 0.001,
    "epochs": 30
}

MODIFIED_CONFIG = {
    "hidden_layers": [32],
    "learning_rate": 0.001,
    "epochs": 30
}

TRAINING_SEEDS = [SEED, SEED + 1, SEED + 2]

print("Baseline configuration:", BASELINE_CONFIG)
print("Modified configuration:", MODIFIED_CONFIG)
print("Training seeds:", TRAINING_SEEDS)


## 2. Diabetes Dataset

This section loads and inspects the diabetes dataset before preprocessing. No values are modified during this initial inspection.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/diabetes.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

# The supplied CSV is headerless, so preserve its first row as data.
df = pd.read_csv(DATA_PATH, header=None)

print("Dataset path:", DATA_PATH)
print("Dataset shape:", df.shape)
print("Column names:", list(df.columns))
display(df.head())

In [ ]:
print("DataFrame information:")
df.info()

print("Descriptive statistics:")
display(df.describe().T)

In [ ]:
print("Missing-value counts:")
display(df.isna().sum().rename("missing_values"))

print("Duplicate-row count:", int(df.duplicated().sum()))

print("Zero-value counts for numeric columns:")
display((df.select_dtypes(include="number") == 0).sum().rename("zero_values"))

In [ ]:
# The headerless dataset uses its final column as the binary target.
TARGET_COLUMN = df.columns[-1]
target_values = sorted(df[TARGET_COLUMN].dropna().unique().tolist())
target_counts = df[TARGET_COLUMN].value_counts().sort_index()
target_percentages = (
    df[TARGET_COLUMN].value_counts(normalize=True).sort_index() * 100
).round(4)

print("Confirmed target column:", TARGET_COLUMN)
print("Target unique values:", target_values)
print("Target class counts:")
display(target_counts.rename("count"))
print("Target class percentages:")
display(target_percentages.rename("percentage"))

### Initial Observations

- The dataset contains 759 rows and 9 columns, giving 8 input features and 1 target column.
- Because the supplied CSV has no header row, the confirmed target is column `8`, the final column, with binary classes `0` and `1`; it represents the diabetes outcome.
- There are no explicit missing values and no duplicate rows.
- Under the standard feature order for this headerless diabetes dataset, suspicious zeros occur in the medical measurement columns for glucose (column `1`: 5), blood pressure (column `2`: 35), skin thickness (column `3`: 224), insulin (column `4`: 371), and BMI (column `5`: 11). The age-like column `7` also contains 63 zeros.
- Zero is valid for the pregnancy-count feature (column `0`: 111) and for the binary target (column `8`: 263). Suspicious zeros will be handled during preprocessing rather than changed during inspection.